# Savage SVG Generation Training (Colab Fast Mode)
This notebook trains Qwen2.5-Coder-0.5B to natively generate SVG code from text prompts. It is optimized to run fully within the free-tier Google Colab T4 GPU time limits!

In [ ]:
!git clone https://github.com/ahmedbarakat207/savage.git
%cd savage
!pip install -q transformers datasets peft trl accelerate cairosvg vtracer hf_transfer torchao>=0.16.1\n%env HF_XET_HIGH_PERFORMANCE=1

## 1. Build Dataset (Fast Mode)
This downloads real images and traces them into high-quality, text-friendly SVGs.
We use `sed` to limit the number of generated images to 1,500 so this step only takes ~1 minute on Colab, rather than 30+ minutes.

In [ ]:
!sed -i 's/target_count=20000/target_count=1000/g' build_dataset.py
!sed -i 's/target_count=5000/target_count=500/g' build_dataset.py
!python build_dataset.py

## 2. Train the Model (Colab Fast Mode)
We use LoRA to fine-tune Qwen2.5-Coder.
The `--colab-fast` flag optimizes memory and context limits specifically for the free 16GB T4 GPU. It halves the sequence length to 2048 and restricts max steps to 200, which guarantees training finishes in under 30 minutes without Out-Of-Memory errors!

In [ ]:
!python train.py --colab-fast

## 3. Generate an SVG
We use the `--engine transformers` flag since Colab uses Linux/CUDA, not Mac MLX.

In [ ]:
!python generate.py --prompt "a beautiful orange cat" --engine transformers --lora_path "./savage-1"

## 4. Render and View SVG
We can display the generated SVG right in the notebook!

In [ ]:
import IPython.display as display
display.display(display.SVG(filename="a_beautiful_orange_cat.svg"))

## 5. Download the Checkpoint
Zip up the LoRA weights and download them.

In [ ]:
!zip -r savage-lora.zip ./savage-1 
try:
    from google.colab import files
    files.download('savage-lora.zip')
except ImportError:
    print("Not on Colab. If on Kaggle, you can download 'savage-lora.zip' from the output panel on the right.")